# 03 — Phân tích độ nhạy trọng số

Notebook chạy thí nghiệm Phase 1.1 trên ba cấu hình trọng số và ba seed. Kết quả
gồm `raw_runs.csv` và `summary.json`, sau đó được lưu sang Google Drive.

In [ ]:
#@title Clone dự án từ GitHub và cài môi trường Colab
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
from pathlib import Path
import shutil
import subprocess
import sys

THU_MUC_COLAB_DRIVE = Path("/content/drive/MyDrive/Genetic_ALO_Colab")
THU_MUC_COLAB_DRIVE.mkdir(parents=True, exist_ok=True)
THU_MUC_DU_AN = Path("/content/genetic-alo")
THU_MUC_KET_QUA_DRIVE = THU_MUC_COLAB_DRIVE / "latest_outputs"
REPOSITORY_URL = "https://github.com/duktrung05/genetic-alo.git"

shutil.rmtree(THU_MUC_DU_AN, ignore_errors=True)
subprocess.run(
    [
        "git", "clone", "--depth", "1", "--branch", "main",
        REPOSITORY_URL, str(THU_MUC_DU_AN),
    ],
    check=True,
)

# Khôi phục output mới nhất từ Drive nếu notebook trước đã tạo kết quả.
if THU_MUC_KET_QUA_DRIVE.is_dir():
    shutil.copytree(
        THU_MUC_KET_QUA_DRIVE,
        THU_MUC_DU_AN / "outputs",
        dirs_exist_ok=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "-r",
        str(THU_MUC_DU_AN / "requirements.txt"),
    ],
    check=True,
)

os.chdir(THU_MUC_DU_AN)
if str(THU_MUC_DU_AN) not in sys.path:
    sys.path.insert(0, str(THU_MUC_DU_AN))

def dong_bo_ket_qua() -> Path:
    """Copy all current outputs to Drive so another notebook can reuse them."""
    THU_MUC_KET_QUA_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copytree(
        THU_MUC_DU_AN / "outputs",
        THU_MUC_KET_QUA_DRIVE,
        dirs_exist_ok=True,
    )
    return THU_MUC_KET_QUA_DRIVE

print(f"✅ Đã clone dự án tại: {THU_MUC_DU_AN}")
print(f"✅ Python: {sys.version.split()[0]}")
subprocess.run(["git", "log", "-1", "--oneline"], cwd=THU_MUC_DU_AN, check=True)
print("✅ Dataset Excel nằm trong data/instances.")

In [ ]:
#@title Chạy phân tích độ nhạy
CHAY_PHAN_TICH_DO_NHAY = True #@param {type:"boolean"}
TEN_DATASET = "easy" #@param ["easy", "medium"]

thu_muc_do_nhay = THU_MUC_DU_AN / "outputs/benchmark/phase1_1_weight_sensitivity"
if CHAY_PHAN_TICH_DO_NHAY:
    subprocess.run(
        [
            sys.executable,
            "scripts/run_phase1_1_sensitivity.py",
            "--input", str(THU_MUC_DU_AN / f"data/instances/instance_{TEN_DATASET}.xlsx"),
            "--output-dir", str(thu_muc_do_nhay),
        ],
        cwd=THU_MUC_DU_AN,
        check=True,
    )
    print(f"✅ Đã đồng bộ sang: {dong_bo_ket_qua()}")
else:
    print("ℹ️ Đã bỏ qua phân tích độ nhạy.")

In [ ]:
#@title Hiển thị bảng tổng hợp
import json
import pandas as pd
from IPython.display import display

summary_path = thu_muc_do_nhay / "summary.json"
if not summary_path.is_file():
    raise FileNotFoundError("Chưa có summary.json. Hãy chạy cell phân tích trước.")

payload = json.loads(summary_path.read_text(encoding="utf-8"))
rows = []
for profile, values in payload["summary"].items():
    rows.append({
        "profile": profile,
        "runs": values["runs"],
        "feasible_runs": values["feasible_runs"],
        "mean_total_soft_score": values["mean_total_soft_score"],
        "mean_runtime_seconds": values["mean_runtime_seconds"],
    })
display(pd.DataFrame(rows))